In [1]:
# Cell 1 - Harmonized data ingestion and source profile audit
from pathlib import Path
import numpy as np
import pandas as pd

DATA_PATH = Path("data/harmonized_master_real.csv")
df_raw = pd.read_csv(DATA_PATH)

source_counts = df_raw["source"].value_counts().to_dict()
target_by_source = df_raw.groupby("source")["target"].agg(["count", "sum", "mean"]).to_dict(orient="index")
missing_by_col = df_raw.isnull().sum()[lambda x: x > 0].to_dict()

print(f"total_rows={len(df_raw)} | total_cols={df_raw.shape[1]}")
print(f"source_counts={source_counts}")
for src, stats in target_by_source.items():
    print(f"source={src} | n={stats['count']} | pos={int(stats['sum'])} | pos_rate={stats['mean']:.4f}")
print(f"missing_features={missing_by_col}")

total_rows=3567 | total_cols=8
source_counts={'CDC_NHANES': 2709, 'UCI_Caracas': 858}
source=CDC_NHANES | n=2709 | pos=20 | pos_rate=0.0074
source=UCI_Caracas | n=858 | pos=55 | pos_rate=0.0641
missing_features={'num_sexual_partners': 791, 'first_sexual_intercourse': 779, 'num_pregnancies': 850, 'smokes': 16, 'hormonal_contraceptives': 525}


In [2]:
# Cell 2 - Dual-purpose feature construction and stratified fold locking
from sklearn.model_selection import StratifiedKFold

df_clean = pd.DataFrame()
df_clean["patient_id"] = [f"PAT_{i+1:04d}" for i in range(len(df_raw))]
df_clean["source"] = df_raw["source"]

# ponytail: MLE numerical representation preserves native floats and explicit NaNs for pipeline imputation
df_clean["age"] = df_raw["age"].astype(float)
df_clean["num_sexual_partners"] = df_raw["num_sexual_partners"].astype(float)
df_clean["first_sexual_intercourse"] = df_raw["first_sexual_intercourse"].astype(float)
df_clean["num_pregnancies"] = df_raw["num_pregnancies"].astype(float)
df_clean["smokes"] = df_raw["smokes"].astype(float)
df_clean["hormonal_contraceptives"] = df_raw["hormonal_contraceptives"].astype(float)
df_clean["target"] = df_raw["target"].astype(int)

# DA human-readable categorical labels for BI/reporting
df_clean["target_label"] = np.where(df_clean["target"] == 1, "Positive", "Negative")
df_clean["age_group"] = pd.cut(
    df_clean["age"], bins=[0, 24, 39, 54, 120], labels=["<25", "25-39", "40-54", "55+"], right=True
)
df_clean["smoker_label"] = df_clean["smokes"].map({1.0: "Smoker", 0.0: "Non-Smoker"}).fillna("Unreported")
df_clean["contraception_label"] = (
    df_clean["hormonal_contraceptives"].map({1.0: "Hormonal User", 0.0: "Non-User"}).fillna("Unreported")
)
df_clean["pregnancy_group"] = (
    pd.cut(df_clean["num_pregnancies"], bins=[-1, 0, 2, 4, 100], labels=["0", "1-2", "3-4", "5+"])
    .astype(str)
    .replace("nan", "Unreported")
)
df_clean["partners_group"] = (
    pd.cut(df_clean["num_sexual_partners"], bins=[0, 1, 4, 1000], labels=["1", "2-4", "5+"])
    .astype(str)
    .replace("nan", "Unreported")
)

# Pre-baked 5-fold Stratified K-Fold (seed=42 locked)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
df_clean["fold_5cv"] = 0
for fold, (tr_idx, va_idx) in enumerate(skf.split(df_clean, df_clean["target"])):
    df_clean.loc[va_idx, "fold_5cv"] = fold + 1

print(f"clean_shape={df_clean.shape}")
print(f"columns={df_clean.columns.tolist()}")
fold_summary = df_clean.groupby("fold_5cv")["target"].agg(["count", "sum"]).to_dict(orient="index")
print(f"fold_summary={fold_summary}")

clean_shape=(3567, 16)
columns=['patient_id', 'source', 'age', 'num_sexual_partners', 'first_sexual_intercourse', 'num_pregnancies', 'smokes', 'hormonal_contraceptives', 'target', 'target_label', 'age_group', 'smoker_label', 'contraception_label', 'pregnancy_group', 'partners_group', 'fold_5cv']
fold_summary={1: {'count': 714, 'sum': 15}, 2: {'count': 714, 'sum': 15}, 3: {'count': 713, 'sum': 15}, 4: {'count': 713, 'sum': 15}, 5: {'count': 713, 'sum': 15}}


In [4]:
# Cell 3 - Quality assurance assertions and master cleaned export
from pathlib import Path

OUT_PATH = Path("data/cervical_cancer_master_cleaned.csv")

assert df_clean.shape == (3567, 16), f"Unexpected shape: {df_clean.shape}"
assert df_clean["patient_id"].nunique() == 3567, "Duplicate patient_id detected"
assert df_clean["target"].sum() == 75, f"Target sum mismatch: {df_clean['target'].sum()}"
assert set(df_clean["fold_5cv"].unique()) == {1, 2, 3, 4, 5}, "Fold ID integrity failure"
for f in range(1, 6):
    f_pos = int(df_clean[df_clean["fold_5cv"] == f]["target"].sum())
    assert f_pos == 15, f"Fold {f} does not have exactly 15 positives (got {f_pos})"

# Save master cleaned dataset
df_clean.to_csv(OUT_PATH, index=False)

# Readback validation to ensure zero serialization drift
df_verified = pd.read_csv(OUT_PATH)
assert df_verified.shape == (3567, 16)
assert df_verified["target"].sum() == 75

da_cols = [c for c in df_verified.columns if c.endswith("_label") or c.endswith("_group")]
mle_cols = ["age", "num_sexual_partners", "first_sexual_intercourse", "num_pregnancies", "smokes", "hormonal_contraceptives", "target", "fold_5cv"]

print(f"master_cleaned_exported={OUT_PATH}")
print(f"file_size_bytes={OUT_PATH.stat().st_size:,}")
print(f"total_records={len(df_verified)} | positive_events={int(df_verified['target'].sum())}")
print(f"da_columns={da_cols}")
print(f"mle_columns={mle_cols}")

master_cleaned_exported=data\cervical_cancer_master_cleaned.csv
file_size_bytes=319,036
total_records=3567 | positive_events=75
da_columns=['target_label', 'age_group', 'smoker_label', 'contraception_label', 'pregnancy_group', 'partners_group']
mle_columns=['age', 'num_sexual_partners', 'first_sexual_intercourse', 'num_pregnancies', 'smokes', 'hormonal_contraceptives', 'target', 'fold_5cv']
